In [6]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("vargr/main_instagram")
print(dataset)
print(dataset['train'].column_names)
print(dataset['train'][0])


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-b6868cb1ddbd5c(…):   0%|          | 0.00/159M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/605868 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sid', 'sid_profile', 'shortcode', 'profile_id', 'date', 'post_type', 'description', 'likes', 'comments', 'username', 'bio', 'following', 'followers', 'num_posts', 'is_business_account', 'lang', 'description_category', 'description_grade', 'image_grade', 'path'],
        num_rows: 605868
    })
})
['sid', 'sid_profile', 'shortcode', 'profile_id', 'date', 'post_type', 'description', 'likes', 'comments', 'username', 'bio', 'following', 'followers', 'num_posts', 'is_business_account', 'lang', 'description_category', 'description_grade', 'image_grade', 'path']
{'sid': 8304346, 'sid_profile': 3496776, 'shortcode': 'Bt5LFpZlm3z', 'profile_id': 2237947779, 'date': '2019-02-15 08:02:35', 'post_type': 1, 'description': 'Laps in spring like conditions. Getting these M29 bikes up to speed. Is it time to go racing yet?\\n.\\n.\\n.\\n.\\n.\\n#intenseforlife #m29 #dh #racing \\n@saddleback_ltd \\n@intensecyclesuk \\n@intensecycles\\n@tld_bike\\n@

In [7]:
df = dataset['train'].to_pandas()

print(df[['likes','followers','date']].head())
print(df.isnull().sum())


   likes  followers                 date
0    150       1204  2019-02-15 08:02:35
1     15       1263  2019-03-19 22:18:58
2     41        139  2019-05-13 19:03:32
3    292      28163  2016-05-07 19:34:45
4    454      27753  2019-04-04 18:52:40
sid                          0
sid_profile                  0
shortcode                    0
profile_id                   0
date                         0
post_type                    0
description                  0
likes                        0
comments                     0
username                     0
bio                     117093
following                    0
followers                    0
num_posts                    0
is_business_account          0
lang                         0
description_category         0
description_grade            0
image_grade                  0
path                         0
dtype: int64


In [8]:
import pandas as pd
import numpy as np

df = dataset['train'].to_pandas()

# Remove accounts with zero followers
df = df[df['followers'] > 0]

# Create engagement rate
df['engagement_rate'] = (df['likes'] + df['comments']) / df['followers']

# Remove extreme outliers (top 1%)
upper_limit = df['engagement_rate'].quantile(0.99)
df = df[df['engagement_rate'] <= upper_limit]

df['engagement_rate'].describe()


,engagement_rate
count,599579.000000
mean,0.111752
std,0.099628
min,0.000000
25%,0.038462
50%,0.083218
75%,0.155556
max,0.613913


In [9]:
low = df['engagement_rate'].quantile(0.33)
high = df['engagement_rate'].quantile(0.66)

def classify(x):
    if x <= low:
        return 0  # low
    elif x <= high:
        return 1  # medium
    else:
        return 2  # high

df['engagement_class'] = df['engagement_rate'].apply(classify)

df['engagement_class'].value_counts()


,count
engagement_class,
2,203854
1,197864
0,197861


In [10]:
df['date'] = pd.to_datetime(df['date'])

df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report



df_sample = df.sample(n=100000, random_state=42)

# Then repeat TFIDF and rest using df_sample

# Text vectorization
tfidf = TfidfVectorizer(max_features=5000)
X_text = tfidf.fit_transform(df_sample['description'])

meta_features = df_sample[['hour',
                    'day_of_week',
                    'followers',
                    'following',
                    'num_posts',
                    'is_business_account']].copy()

# Convert boolean to int
meta_features['is_business_account'] = meta_features['is_business_account'].astype(int)

# Ensure all numeric
meta_features = meta_features.astype(float)

meta_features = meta_features.values

# Combine
import scipy.sparse as sp
X = sp.hstack([X_text, meta_features])

y = df_sample['engagement_class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

preds = model.predict(X_test)

print(classification_report(y_test, preds))


              precision    recall  f1-score   support

           0       0.65      0.66      0.66      6700
           1       0.45      0.37      0.41      6595
           2       0.60      0.69      0.64      6705

    accuracy                           0.58     20000
   macro avg       0.57      0.58      0.57     20000
weighted avg       0.57      0.58      0.57     20000



In [12]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, n_jobs=-1)
model.fit(X_train, y_train)

preds = model.predict(X_test)

print(classification_report(y_test, preds))


              precision    recall  f1-score   support

           0       0.65      0.57      0.61      6700
           1       0.43      0.26      0.32      6595
           2       0.52      0.79      0.63      6705

    accuracy                           0.54     20000
   macro avg       0.54      0.54      0.52     20000
weighted avg       0.54      0.54      0.52     20000



In [13]:
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,2),
    min_df=5
)


In [14]:
df_sample['followers'] = np.log1p(df_sample['followers'])
df_sample['following'] = np.log1p(df_sample['following'])
df_sample['num_posts'] = np.log1p(df_sample['num_posts'])


In [15]:
y_train.value_counts(normalize=True)


,proportion
engagement_class,
2,0.338537
0,0.332275
1,0.329188


In [16]:
!pip install xgboost
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    tree_method='hist'
)

model.fit(X_train, y_train)

preds = model.predict(X_test)

from sklearn.metrics import classification_report
print(classification_report(y_test, preds))


              precision    recall  f1-score   support

           0       0.68      0.63      0.65      6700
           1       0.45      0.40      0.43      6595
           2       0.59      0.71      0.65      6705

    accuracy                           0.58     20000
   macro avg       0.58      0.58      0.58     20000
weighted avg       0.58      0.58      0.58     20000



In [17]:
!pip install transformers torch

In [18]:
from transformers import DistilBertTokenizer, DistilBertModel
import torch

In [19]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained('distilbert-base-uncased')

model.eval()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [20]:
def get_embedding(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

    with torch.no_grad():
        outputs = model(**inputs)

    return outputs.last_hidden_state[:,0,:].numpy()

In [21]:
sample_df = df.sample(5000)

In [22]:
embeddings = sample_df['description'].apply(get_embedding)

In [23]:
import numpy as np

X_text = np.vstack(embeddings.values)

In [25]:
meta_features = sample_df[
    ['followers', 'following', 'num_posts', 'hour', 'day_of_week']
].values

In [26]:
X = np.hstack([X_text, meta_features])

In [27]:
from xgboost import XGBClassifier

model = XGBClassifier()

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [28]:
from sklearn.metrics import classification_report

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.68      0.63      0.65      6700
           1       0.45      0.40      0.43      6595
           2       0.59      0.71      0.65      6705

    accuracy                           0.58     20000
   macro avg       0.58      0.58      0.58     20000
weighted avg       0.58      0.58      0.58     20000

